<a href="https://colab.research.google.com/github/manasviparam/ML_Practical/blob/main/ML_EXP8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
TARGET_COL = "PlacementStatus"
DATA_PATH = "/content/drive/MyDrive/Colab/placement_predict_50k_adjusted.csv"

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
df_check = pd.read_csv(DATA_PATH, nrows=5)
print(df_check.columns.tolist())
print(df_check.shape)

['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly']
(5, 21)


In [6]:
df = pd.read_csv(DATA_PATH)

if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])

y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le

imputer = SimpleImputer(strategy="median")
X[num_cols] = imputer.fit_transform(X[num_cols])

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

In [7]:
VAL_SIZE = 0.15
TEST_SIZE = 0.15

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio,
    stratify=y_train_val, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Train: (34999, 19) | Val: (7501, 19) | Test: (7500, 19)


In [8]:
def boosting_benchmark(X_train, y_train, X_val, y_val):
    results = []

    # ---- AdaBoost ----
    ada_base = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)
    ada = AdaBoostClassifier(
        estimator=ada_base,
        n_estimators=200,
        learning_rate=0.5,
        random_state=RANDOM_STATE,
    )

    t0 = time.time()
    ada.fit(X_train, y_train)
    ada_fit_time = time.time() - t0

    ada_val_pred = ada.predict(X_val)
    ada_val_proba = ada.predict_proba(X_val)[:, 1]

    results.append({
        "model": "AdaBoost",
        "val_accuracy": accuracy_score(y_val, ada_val_pred),
        "val_f1": f1_score(y_val, ada_val_pred),
        "val_roc_auc": roc_auc_score(y_val, ada_val_proba),
        "best_n_estimators": ada.n_estimators,
        "fit_time_sec": round(ada_fit_time, 2),
    })

    # ---- XGBoost with early stopping ----
    xgb = XGBClassifier(
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    t0 = time.time()
    xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    xgb_fit_time = time.time() - t0

    xgb_val_pred = xgb.predict(X_val)
    xgb_val_proba = xgb.predict_proba(X_val)[:, 1]

    results.append({
        "model": "XGBoost",
        "val_accuracy": accuracy_score(y_val, xgb_val_pred),
        "val_f1": f1_score(y_val, xgb_val_pred),
        "val_roc_auc": roc_auc_score(y_val, xgb_val_proba),
        "best_n_estimators": xgb.best_iteration + 1,
        "fit_time_sec": round(xgb_fit_time, 2),
    })

    return pd.DataFrame(results).sort_values(
        "val_accuracy", ascending=False
    ).reset_index(drop=True)

In [9]:
leaderboard = boosting_benchmark(X_train, y_train, X_val, y_val)
print("\nValidation leaderboard (sorted by val_accuracy):")
print(leaderboard.to_string(index=False))

leaderboard.to_csv("boosting_benchmark_results.csv", index=False)
print("\nSaved results to boosting_benchmark_results.csv")


Validation leaderboard (sorted by val_accuracy):
   model  val_accuracy   val_f1  val_roc_auc  best_n_estimators  fit_time_sec
AdaBoost      0.796294 0.780963     0.880162                200         14.08
 XGBoost      0.795894 0.782744     0.882638                135          1.03

Saved results to boosting_benchmark_results.csv


In [11]:
from google.colab import files
files.download("boosting_benchmark_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
best_model_name = leaderboard.iloc[0]["model"]
print(f"Best model on validation set: {best_model_name}")

if best_model_name == "AdaBoost":
    best_model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE),
        n_estimators=200,
        learning_rate=0.5,
        random_state=RANDOM_STATE,
    )
    best_model.fit(X_train, y_train)
else:
    best_model = XGBClassifier(
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

test_pred = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]

print("\nFinal test-set performance:")
print(f"Accuracy: {accuracy_score(y_test, test_pred):.4f}")
print(f"F1:       {f1_score(y_test, test_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, test_proba):.4f}")

Best model on validation set: AdaBoost

Final test-set performance:
Accuracy: 0.7891
F1:       0.7748
ROC-AUC:  0.8760


In [13]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred))

[[3196  742]
 [ 840 2722]]
              precision    recall  f1-score   support

           0       0.79      0.81      0.80      3938
           1       0.79      0.76      0.77      3562

    accuracy                           0.79      7500
   macro avg       0.79      0.79      0.79      7500
weighted avg       0.79      0.79      0.79      7500

